# Restaurant Vibe Classifier - Dataset Construction

This notebook performs the dataset engineering pipeline for the restaurant vibe classification NLP project.

Pipeline stages:
- Yelp dataset loading
- dataset exploration
- null value inspection
- category analysis
- restaurant filtering
- geographic filtering
- restaurant-review merging
- supervised dataset construction

## Import Libraries

Import required libraries for data loading, preprocessing, filtering, and exporting datasets.

In [3]:
import pandas as pd
import json
from pathlib import Path

In [4]:
VIBE_LABELS = [
    "date_night",
    "study_spot",
    "casual_hangout",
    "family_friendly",
    "upscale",
    "trendy",
    "late_night",
]

## Define File Paths & Load Data

Define paths to the Yelp Open Dataset business and review JSON files.

In [5]:
RAW_DIR = Path("../data/raw")

business_path = RAW_DIR / "yelp_businesses.json"
reviews_path = RAW_DIR / "yelp_reviews.json"

# load
businesses = pd.read_json(business_path, lines=True)
reviews = pd.read_json(reviews_path, lines=True)

## Dataset Inspection

Inspect dataset dimensions and available columns to understand dataset structure.

In [6]:
# look at the raw data 
print("Businesses:", businesses.shape)
print("Reviews:", reviews.shape)

print("\nBusiness columns:")
print(businesses.columns.tolist())

print("\nReview columns:")
print(reviews.columns.tolist())

Businesses: (150346, 14)
Reviews: (6990280, 9)

Business columns:
['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

Review columns:
['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']


In [7]:
print("\nReviews Head")
reviews.head()


Reviews Head


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [8]:
print("\Businesses Head")
businesses.head()

\Businesses Head


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


## Null Value Analysis

Check for missing values across datasets to identify potential preprocessing requirements.

In [9]:
# check missing values 
print("Null values in Businesses")
businesses.isnull().sum().sort_values(ascending=False)

Null values in Businesses


hours           23223
attributes      13744
categories        103
address             0
business_id         0
name                0
postal_code         0
state               0
city                0
latitude            0
review_count        0
stars               0
longitude           0
is_open             0
dtype: int64

In [10]:
print("Null values in Reviews")
reviews.isnull().sum().sort_values(ascending=False)

Null values in Reviews


review_id      0
user_id        0
business_id    0
stars          0
useful         0
funny          0
cool           0
text           0
date           0
dtype: int64

## Pre-Processing Restaurant Categories

1. isolate the categories column in the businesses dataset and drop any null values 
2. extract all unique categories 
3. count how many times the category label is used 
4. export data into `data\processed\category_counts.csv`
5. identify category labels that imply the business is a food establishment that we can use later to filter the data with

In [11]:
from collections import Counter
import pandas as pd
from pathlib import Path

# drop null values
category_series = businesses["categories"].dropna()

# split categories on the comma
split_categories = category_series.str.split(",")

# flatten and clean whitespace
all_categories = [
    cat.strip()
    for sublist in split_categories
    for cat in sublist
]

# unique categories
category_counts = Counter(all_categories)

# convert to df
category_df = pd.DataFrame(
    category_counts.items(),
    columns=["category", "count"]
).sort_values(by="count", ascending=False)

# save counts to csv file
output_path = Path("../data/processed/category_counts.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

category_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(category_df.head(20))

Saved to: ..\data\processed\category_counts.csv
                      category  count
17                 Restaurants  52268
18                        Food  27781
12                    Shopping  24395
135              Home Services  14356
107              Beauty & Spas  14292
38                   Nightlife  12281
4             Health & Medical  11890
7               Local Services  11198
36                        Bars  11065
50                  Automotive  10773
90   Event Planning & Services   9895
26                  Sandwiches   8366
37      American (Traditional)   8139
83                 Active Life   7687
80                       Pizza   7093
20                Coffee & Tea   6703
25                   Fast Food   6472
43          Breakfast & Brunch   6239
145             American (New)   6097
58             Hotels & Travel   5857


## Restaurant Filtering

Filter the dataset to restaurant, cafe, food, and nightlife-related businesses.

In [12]:
FOOD_CATEGORIES = [
    "Restaurants",
    "Food",
    "Bars",
    "Nightlife",
    "Coffee & Tea",
    "Breakfast & Brunch",
    "Pizza",
    "Fast Food",
    "Burgers",
    "Mexican",
    "Italian",
    "Seafood",
    "Desserts",
    "Chinese",
    "Bakeries",
    "Salad",
    "Chicken Wings",
    "Cafes",
    "Delis",
    "Sports Bars",
    "Japanese",
    "Pubs",
    "Cocktail Bars",
    "Sushi Bars",
    "Barbeque",
    "Juice Bars & Smoothies",
    "Ice Cream & Frozen Yogurt",
    "Specialty Food",
    "Sandwiches"
]

restaurant_mask = businesses["categories"].fillna("").apply(
    lambda x: any(cat in x for cat in FOOD_CATEGORIES)
)

restaurants = businesses[restaurant_mask].copy()

print(restaurants.shape)
restaurants.head(10)

(67742, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
12,il_Ro8jwPlHresjw9EGmBg,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,1,"{'RestaurantsReservations': 'False', 'Restaura...","American (Traditional), Restaurants, Diners, B...","{'Monday': '6:0-22:0', 'Tuesday': '6:0-22:0', ..."
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,0,"{'OutdoorSeating': 'False', 'RestaurantsGoodFo...","Food, Delis, Italian, Bakeries, Restaurants","{'Monday': '10:0-18:0', 'Tuesday': '10:0-20:0'..."
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."


In [13]:
# make sure that "food" doesnt also accidentally include grocery stores, drugstores, etc. 
EXCLUDED_CATEGORIES = [
    "Adult Entertainment",
    "Strip Clubs",
    "Music Venues",
    "Musicians",
    "Tobacco Shops",
    "Shopping",
    "Beer, Wine & Spirits",
    "Convenience Stores",
    "Gas Stations",
    "Florists",
    "Beauty & Spas",
    "Hotels"
]
print(restaurants.shape)

restaurants = restaurants[
    ~restaurants["categories"].fillna("").apply(
        lambda x: any(cat in x for cat in EXCLUDED_CATEGORIES)
    )
]


print(restaurants.shape)
restaurants.head(10)

(67742, 14)
(59165, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","Vietnamese, Food, Restaurants, Food Trucks","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
12,il_Ro8jwPlHresjw9EGmBg,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,1,"{'RestaurantsReservations': 'False', 'Restaura...","American (Traditional), Restaurants, Diners, B...","{'Monday': '6:0-22:0', 'Tuesday': '6:0-22:0', ..."
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,2575 E Bay Dr,Largo,FL,33771,27.916116,-82.760461,4.5,100,0,"{'OutdoorSeating': 'False', 'RestaurantsGoodFo...","Food, Delis, Italian, Bakeries, Restaurants","{'Monday': '10:0-18:0', 'Tuesday': '10:0-20:0'..."
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."


In [14]:
restaurants[["name", "city", "state", "categories", "review_count"]].sample(10)

,name,city,state,categories,review_count
38477,Steak ’n Shake,Tampa,FL,"American (New), Burgers, Diners, Restaurants, ...",82
112295,Cali's Coffee & Creamery,Smyrna,TN,"Food, Ice Cream & Frozen Yogurt, Coffee & Tea,...",39
44495,Harry C's Craft House,Indianapolis,IN,"Bars, Breakfast & Brunch, Nightlife, Restauran...",16
80367,Taco Bell,Brentwood,TN,"Restaurants, Tex-Mex, Mexican, Breakfast & Bru...",16
27254,The BrickHaus,Ridley Park,PA,"Food, Coffee & Tea, Cafes, Breakfast & Brunch,...",70
36497,McDonald's,Tucson,AZ,"Coffee & Tea, Restaurants, Fast Food, Burgers,...",28
143426,Starbucks,Franklin,TN,"Food, Coffee & Tea",16
31536,Cafe and Salad Works,Camden,NJ,"Vietnamese, Restaurants",11
906,Brew House Pub & Grill,Reno,NV,"Restaurants, American (Traditional), Bars, Nig...",6
1572,Old Country Buffet,Fairless Hills,PA,"Restaurants, American (Traditional), Breakfast...",27


## Geographic Filtering

Restrict the dataset to the Santa Barbara region:
- Santa Barbara
- Goleta
- Carpinteria

In [15]:
TARGET_CITIES = [
    "Santa Barbara",
    "Goleta",
    "Carpinteria"
]

sb_restaurants = restaurants[
    (restaurants["state"] == "CA") &
    (restaurants["city"].isin(TARGET_CITIES))
].copy()

print(sb_restaurants.shape)

(1275, 14)


In [16]:
sb_restaurants[
    ["name", "city", "state", "categories", "stars", "review_count"]
].sample(20)

,name,city,state,categories,stars,review_count
132301,Yona Redz,Santa Barbara,CA,"Restaurants, Latin American, Mexican, Tacos",4.5,125
143854,Violette Bakeshop,Santa Barbara,CA,"Food, Bakeries",5.0,15
87857,Panera Bread,Santa Barbara,CA,"Bagels, Sandwiches, Soup, Restaurants, Bakerie...",2.5,176
80310,Belcampo Meat Co Restaurant,Santa Barbara,CA,"Food, Meat Shops, American (New), Burgers, Bre...",4.0,33
63789,Lunchbox,Santa Barbara,CA,"Food Delivery Services, Food, Event Planning &...",5.0,11
81804,Yen Ching Restaurant,Santa Barbara,CA,"Buffets, Restaurants, Chinese",2.5,44
100472,Test Pilot,Santa Barbara,CA,"Nightlife, Bars, Tiki Bars, Cocktail Bars",4.5,252
41593,Chuck's of Hawaii,Santa Barbara,CA,"Salad, Steakhouses, Restaurants, Nightlife, Se...",3.5,260
58549,Santa Barbara Brewing Co,Santa Barbara,CA,"Breweries, Restaurants, Sports Bars, Nightlife...",3.5,642
40208,Derf's Cafe,Santa Barbara,CA,"Restaurants, American (Traditional)",3.5,126


## Merge Restaurants with Reviews

Merge Yelp reviews with restaurant metadata using the `business_id` field.

In [17]:
# grab all reviews and match them to the business they are reviewing 
sb_reviews = reviews.merge(
    sb_restaurants[
        [
            "business_id",
            "name",
            "city",
            "categories",
            "stars",
            "review_count"
        ]
    ],
    on="business_id",
    how="inner"
)

print(sb_reviews.shape)

(200040, 14)


In [18]:
# let's take a look at what the merged data looks like
# sort by name so if multiple rows for the same restaurant they are grouped together

sb_reviews[["name", "city", "categories", "text"]].sort_values(by=['name']).sample(30).sort_values(by=['name'])

,name,city,categories,text
6703,Cal Taco,Goleta,"Burgers, Cafes, Restaurants, Mexican, American...","Under new ownership, prices went up, quality i..."
2784,Cold Spring Tavern,Santa Barbara,"American (Traditional), Restaurants, Bars, Nig...",What a cool spot hidden in the mountains. Trip...
158968,D'Angelo's Bakery,Santa Barbara,"Restaurants, Food, Breakfast & Brunch, Bakeries",The breakfast was ok but the fresh baked bread...
187579,Elephant Bar Restaurant,Goleta,"Restaurants, Bars, American (New), Asian Fusio...",This place saved a buddy and me on a Thursday ...
129989,Foodland,Santa Barbara,"International Grocery, Fruits & Veggies, Meat ...",Great store with a very friendly staff. Best p...
192396,Jane At The Marketplace,Goleta,"Pizza, Wine Bars, Restaurants, Venues & Event ...",This was my second visit to Jane. We came in ...
165870,Julienne,Santa Barbara,"Restaurants, American (New), Seafood, French",This is one of our go to places when we are go...
81859,Kyle's Kitchen,Santa Barbara,"Cafes, Restaurants, Sandwiches, Chicken Shop, ...",Just went here for the first time for dinner w...
166909,LOKUM Turkish Delight & Baklava,Santa Barbara,"Coffee & Tea, Food, Desserts, Turkish, Restaur...","Overall, pretty friggin' dope spot. \n\nIt's ..."
134152,Longboard's Grill,Santa Barbara,"Nightlife, Seafood, American (Traditional), Ba...",After a long day driving down Big Sur from Mon...


## NLP Input Construction

Create combined NLP inputs using restaurant categories and Yelp review text.

In [19]:
dataset_df = sb_reviews[
    ["name", "city", "categories", "text"]
].copy()

dataset_df["input_text"] = (
    "Categories: " +
    dataset_df["categories"].fillna("") +
    " | Review: " +
    dataset_df["text"].fillna("")
)

In [20]:
dataset_df['input_text'].head(10)

0    Categories: Steakhouses, Sushi Bars, Restauran...
1    Categories: Restaurants, Sushi Bars | Review: ...
2    Categories: Fast Food, Burgers, Restaurants | ...
3    Categories: Food, Restaurants, Salad, Coffee &...
4    Categories: Restaurants, Fast Food, Mexican, T...
5    Categories: Restaurants, Thai | Review: This i...
6    Categories: Food, Breweries | Review: Sweet sp...
7    Categories: American (Traditional), American (...
8    Categories: Health Markets, Coffee & Tea, Ice ...
9    Categories: Restaurants, Cafes, Breakfast & Br...
Name: input_text, dtype: str

# Example Input–Output Pairs

| Input Text | Output Label |
|---|---|
| "Categories: Coffee & Tea, Cafes \| Review: Quiet cafe with students working on laptops." | study_spot |
| "Categories: Cocktail Bars, Nightlife \| Review: Stylish cocktail lounge with loud music and drinks." | late_night |
| "Categories: Seafood, Wine Bars \| Review: Elegant dinner experience with wine pairings." | upscale |
| "Categories: Mexican, Restaurants \| Review: Relaxed taco spot with quick service." | casual_hangout |

## Export Candidate Labeling Dataset

Export processed review examples for supervised vibe labeling and balancing.

In [21]:
dataset_df[
    ["input_text"]
].to_csv(
    "../data/processed/input_text_all_entries.csv",
    index=False
)